# 03 - Fixed-Partner PPO Retraining Diagnostic

Post-training diagnostic that holds one side of a trained joint policy fixed and retrains the other side with PPO under a fixed compute budget.

**Computes:**
- Original self-play return, measured as undiscounted episodic team return
- Return after fixed-budget PPO retraining against the fixed partner
- Retraining gap, measured as the retrained return minus the original self-play return

This diagnostic is a budget-limited PPO retraining check under the tested policy class and seed protocol. Its reported quantity is the retraining gap between the retrained-policy return and the original self-play return.

In [ ]:
# =========================================================
# POST-TRAINING FIXED-PARTNER PPO RETRAINING DIAGNOSTIC
# Parallel, resume-safe diagnostic for the final paper method labels.
# =========================================================

import os, csv, random
import numpy as np
import gymnasium as gym
import torch
from joblib import Parallel, delayed

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.utils import set_random_seed

from overcooked_ai_py.mdp.overcooked_env import OvercookedEnv, OvercookedGridworld
from overcooked_ai_py.mdp.actions import Action

# ---- MATCH TRAINING CONFIG ----
LAYOUT = "asymmetric_advantages"  # main paper layout
HORIZON = 400
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RUNS_DIR = "/content/drive/MyDrive/runs"

NUM_ACTIONS = len(Action.ALL_ACTIONS)

# Output file for fixed-partner retraining diagnostic results.
RETRAIN_RESULTS_CSV = "/content/drive/MyDrive/fixed_partner_retraining_results.csv"
EVAL_EPISODES = 20  # undiscounted diagnostic rollouts
RETRAIN_STEPS = 200_000  # fixed-budget PPO retraining steps

BASELINES = ["Baseline", "PPO+LLM", "HARL", "PBT_PPO"]
ENV_NAMES = ["No Noise", "Noise", "Delay", "Combo"]
SEEDS = [1001, 2002, 3003, 4004, 5005]


# =========================================================
# MLAM PREWARM
# =========================================================
def warmup_mlam():
    """
    Ensures MediumLevelActionManager pickle is built once in the main process
    before workers spawn.
    """
    print("Prewarming MLAM planner...")
    mdp = OvercookedGridworld.from_layout_name(LAYOUT)
    env = OvercookedEnv.from_mdp(mdp, horizon=HORIZON)

    # This call forces MLAM creation.
    _ = env.featurize_state_mdp(env.state)

    print("MLAM prewarm complete.")


# =========================================================
# ENVIRONMENT WRAPPERS
# =========================================================
class OCWrapper(gym.Env):

    def __init__(self, layout):
        super().__init__()
        mdp = OvercookedGridworld.from_layout_name(layout)
        self.oc = OvercookedEnv.from_mdp(mdp, horizon=HORIZON)

        o0, _ = self.oc.featurize_state_mdp(self.oc.state)
        self.obs_shape = o0.flatten().shape

        self.observation_space = gym.spaces.Box(-np.inf, np.inf, self.obs_shape, dtype=np.float32)
        self.action_space = gym.spaces.MultiDiscrete([NUM_ACTIONS, NUM_ACTIONS])

    def _obs(self):
        o0, _ = self.oc.featurize_state_mdp(self.oc.state)
        return o0.flatten().astype(np.float32)

    def reset(self, seed=None, options=None):
        if seed is not None:
            random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
            if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        self.oc.reset()
        return self._obs(), {}

    def step(self, action):
        a0, a1 = int(action[0]), int(action[1])
        joint = [Action.ALL_ACTIONS[a0], Action.ALL_ACTIONS[a1]]
        _, r, done, info = self.oc.step(joint)
        return self._obs(), float(r), bool(done), False, info


class OCWrapperNoise(OCWrapper):
    def step(self, action):
        obs, r, term, trunc, info = super().step(action)
        return obs + np.random.normal(0,0.01, obs.shape), r, term, trunc, info


class OCWrapperDelay(OCWrapper):
    def __init__(self, layout, noise_prob=0.2, delay_penalty=0.5):
        super().__init__(layout)
        self.noise_prob = noise_prob
        self.delay_penalty = delay_penalty
    def step(self, action):
        obs, r, term, trunc, info = super().step(action)
        if np.random.rand() < self.noise_prob: r -= self.delay_penalty
        return obs, r, term, trunc, info


class OCWrapperCombo(OCWrapper):
    def __init__(self, layout, noise_prob=0.2, delay_penalty=0.5):
        super().__init__(layout)
        self.noise_prob = noise_prob
        self.delay_penalty = delay_penalty
    def step(self, action):
        obs, r, term, trunc, info = super().step(action)
        obs = obs + np.random.normal(0,0.01, obs.shape)
        if np.random.rand() < self.noise_prob: r -= self.delay_penalty
        return obs, r, term, trunc, info


def make_env(env_name, layout):
    m = {
        "no noise": OCWrapper,
        "noise": OCWrapperNoise,
        "delay": OCWrapperDelay,
        "combo": OCWrapperCombo,
    }
    return Monitor(m[env_name.lower()](layout))


# =========================================================
# SELF-PLAY RETURN EVALUATION
# =========================================================

def set_global_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    set_random_seed(seed)

def eval_joint_policy(model, env, episodes=EVAL_EPISODES):
    scores = []
    for _ in range(episodes):
        obs, _ = env.reset()
        ep = 0
        done = False
        while not done:
            a, _ = model.predict(obs, deterministic=True)
            obs, r, term, trunc, _ = env.step(a)
            done = term or trunc
            ep += r
        scores.append(ep)
    return float(np.mean(scores)), float(np.std(scores))


# =========================================================
# FIXED-PARTNER RETRAINING ENVIRONMENT
# =========================================================
class OvercookedFixedPartnerEnv(gym.Env):
    def __init__(self, env_name, layout, partner_model, agent_idx):
        super().__init__()
        self.agent_idx = agent_idx
        self.partner_idx = 1 - agent_idx
        self.partner_model = partner_model

        self.base_env = make_env(env_name, layout).env
        self.observation_space = self.base_env.observation_space
        self.action_space = gym.spaces.Discrete(NUM_ACTIONS)
        self._last_obs = None

    def reset(self, seed=None, options=None):
        if seed is not None:
            set_global_seed(seed)
        obs, info = self.base_env.reset(seed=seed)
        self._last_obs = obs
        return obs, info

    def step(self, action):
        a_self = int(action)
        partner_joint, _ = self.partner_model.predict(self._last_obs, deterministic=True)
        a_partner = int(partner_joint[self.partner_idx])

        joint = np.zeros(2, dtype=np.int64)
        joint[self.agent_idx] = a_self
        joint[self.partner_idx] = a_partner

        obs, r, term, trunc, info = self.base_env.step(joint)
        self._last_obs = obs
        return obs, float(r), bool(term), bool(trunc), info


# =========================================================
# FIXED-BUDGET PPO RETRAINING
# =========================================================
def train_fixed_partner_policy(env_name, layout, partner_model, agent_idx, seed):
    set_global_seed(seed)
    retrain_env = Monitor(OvercookedFixedPartnerEnv(env_name, layout, partner_model, agent_idx))

    retrained = PPO("MlpPolicy", retrain_env,
             n_steps=2048, batch_size=2048,
             learning_rate=3e-4, gamma=0.99,
             verbose=0, device=DEVICE, seed=seed)

    retrained.learn(total_timesteps=RETRAIN_STEPS)

    scores = []
    for _ in range(EVAL_EPISODES):
        obs, _ = retrain_env.reset()
        ep = 0
        done = False
        while not done:
            a, _ = retrained.predict(obs, deterministic=True)
            obs, r, term, trunc, _ = retrain_env.step(a)
            done = term or trunc
            ep += r
        scores.append(ep)
    return float(np.mean(scores)), float(np.std(scores))


def compute_fixed_partner_retraining_for_model(baseline, env_name, seed):

    safe_base = baseline.replace(" ", "_")
    safe_env = env_name.replace(" ", "_")
    model_path = f"{RUNS_DIR}/{safe_base}/{safe_env}/seed_{seed}/final_model.zip"

    if not os.path.exists(model_path):
        print("Missing:", model_path)
        return None

    model = PPO.load(model_path, device=DEVICE)

    env_self = make_env(env_name, LAYOUT)
    v_self_m, v_self_s = eval_joint_policy(model, env_self)

    v_retrain_m, v_retrain_s = train_fixed_partner_policy(env_name, LAYOUT, model, 0, seed+999)
    retraining_gap = v_retrain_m - v_self_m

    return {
        "baseline": baseline,
        "env": env_name,
        "seed": seed,
        "V_self_mean": v_self_m,
        "V_self_std": v_self_s,
        "V_retrain_mean": v_retrain_m,
        "V_retrain_std": v_retrain_s,
        "delta": retraining_gap,
        "retraining_gap": retraining_gap,
    }


# =========================================================
# CSV RESUME LOGIC
# =========================================================
def load_completed_set():
    done = set()
    if os.path.exists(RETRAIN_RESULTS_CSV):
        with open(RETRAIN_RESULTS_CSV) as f:
            next(f)
            for row in csv.reader(f):
                done.add((row[0], row[1], row[2]))
    return done


# =========================================================
# MAIN (PARALLEL + RESUME)
# =========================================================
if __name__ == "__main__":

    warmup_mlam()

    if not os.path.exists(RETRAIN_RESULTS_CSV):
        with open(RETRAIN_RESULTS_CSV, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                "baseline","env","seed",
                "V_self_mean","V_self_std",
                "V_retrain_mean","V_retrain_std","delta"
            ])

    completed = load_completed_set()

    jobs = [(b, e, s) for b in BASELINES for e in ENV_NAMES for s in SEEDS
            if (b, e, str(s)) not in completed]

    print("Remaining fixed-partner retraining diagnostic jobs:", len(jobs))

    def wrapper(job):
        res = compute_fixed_partner_retraining_for_model(*job)
        if res is None: return None

        with open(RETRAIN_RESULTS_CSV, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                res["baseline"], res["env"], res["seed"],
                round(res["V_self_mean"],2),
                round(res["V_self_std"],2),
                round(res["V_retrain_mean"],2),
                round(res["V_retrain_std"],2),
                round(res["delta"],2),
            ])
        print("Saved fixed-partner retraining diagnostic result:", job)
        return res

    Parallel(n_jobs=6)(delayed(wrapper)(job) for job in jobs)

    print("Done. Saved fixed-partner retraining diagnostic results to:", RETRAIN_RESULTS_CSV)